#Initialization

In [0]:
%python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

#Read bronze table crm_prd_info

In [0]:
%python
df = spark.table("workspace.bronze.crm_prd_info")

In [0]:
%python
df.display()

#Silver Transformations

##Trimming

In [0]:
%python
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Cost Cleanup

In [0]:
%python
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

##Product key parsing

In [0]:
%python
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

##Product Line Normalization

In [0]:
%python
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
        .when(F.upper(col("prd_line")) == "R", "Road")
        .when(F.upper(col("prd_line")) == "T", "Touring")
        .when(F.upper(col("prd_line")) == "S", "Other Sales")
        .otherwise("n/a")
    )
)

## Date Casting

In [0]:
%python
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

##Renaming columns

In [0]:
%python
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
%python
df.limit(10).display()

#Writing Silver Table

In [0]:
%python
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanity checks of silver table

In [0]:
SELECT * FROM workspace.silver.crm_products LIMIT 10